Install Dependencies

In [1]:
pip install openai pillow imghdr

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement imghdr (from versions: none)
ERROR: No matching distribution found for imghdr


Load and Prepare the Image

Send Image to OpenAI for Parsing

Structure Output into JSON

In [2]:
from openai import OpenAI
from IPython.display import Image, display
import imghdr
import base64

client = OpenAI()
def get_billing_details_normal(image_path):
    
    with open(image_path, "rb") as f:
        img_bytes = f.read()
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")
        
    response = client.chat.completions.create(
            model="gpt-4o-mini",
            max_tokens=300,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": "Extract billing details (invoice number, date, total, line items) from this image."
                        },
                        # The content type should be "image_url" to use gpt-4-turbo's vision capabilities
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/{imghdr.what(image_path)};base64,{img_b64}"
                            }
                        },
                    ],
                 }
                ]
     )
    
    parsed_data = response.choices[0].message.content
    print(parsed_data)

C:\Users\panka\AppData\Local\Temp\ipykernel_34796\2401332957.py:3: DeprecationWarning: 'imghdr' is deprecated and slated for removal in Python 3.13
  import imghdr


In [ ]:
def calculate_total(billing_json):
    return sum(item["quantity"] * item["price"] for item in billing_json["items"])

In [3]:
def get_billing_details_extract_custom(image_path):
   
    with open(image_path, "rb") as f:
        img_bytes = f.read()
    img_b64 = base64.b64encode(img_bytes).decode("utf-8")

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        max_tokens=300,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                         "text": """Extract billing details in JSON format:
                        {
                          "invoice_number": "",
                          "date": "",
                          "total": "",
                          "line items": "",
                          "items": [{"description": "", "quantity": "", "price": ""}]
                        }"""
                    },
                    # The content type should be "image_url" to use gpt-4-turbo's vision capabilities
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/{imghdr.what(image_path)};base64,{img_b64}"
                        }
                    },
                ],
             }
            ]
        )
    
    billing_json = response.choices[0].message.content
    print(billing_json)

In [4]:
def get_billing_custom(image_path):
    img = Image(url=filename)
    display(img)
    print("_____________________Parse Image________________________")
    get_billing_details_normal(filename)
    print("_____________________Parse Image into formattted_________________________")
    get_billing_details_extract_custom(filename)

In [5]:
print("_____________________Sample Invoice Details________________________")
filename="sampleinvoice.png"
get_billing_custom(filename)
print("_____________________Handwritten Invoice Details________________________")
filename="handwritteninvoice.png"
get_billing_custom(filename)

_____________________Sample Invoice Details________________________


_____________________Parse Image________________________
Here are the extracted billing details from the invoice:

- **Invoice Number:** INV-10012
- **Date Issued:** 26/03/2021
- **Due Date:** 25/04/2021
- **Total:** $1,699.48
- **Line Items:**
  - **Description:** Services
    - **Rate:** $55.00
    - **Quantity:** 10
    - **Amount:** $550.00
  - **Description:** Consulting
    - **Rate:** $75.00
    - **Quantity:** 15
    - **Amount:** $1,125.00
  - **Description:** Materials
    - **Rate:** $123.39
    - **Quantity:** 1
    - **Amount:** $123.39
- **Subtotal:** $1,798.39
- **Discount:** -$179.84
- **Tax:** +$80.93
- **Deposit Due:** $169.95
_____________________Parse Image into formattted_________________________
Here are the billing details extracted in JSON format:

```json
{
  "invoice_number": "INV-10012",
  "date": "26/3/2021",
  "total": "$1,699.48",
  "line items": 3,
  "items": [
    {
      "description": "Services",
      "quantity": "10",
      "price": "$550.00"
    },


_____________________Parse Image________________________
Here are the extracted billing details from the image:

- **Invoice Number:** Not explicitly mentioned
- **Date:** May 1814
- **Total:** £327 15s 2d
- **Line Items:**
  - Jaconet: 67
  - White Mull: 90
  - Fancy White Muslin: 26
  - Jonguil D: 28
  - Coquelicot D: 55
  - Case: £2 5s
  - Insurance: £5 19s 2d
  - Carriage to Hull: £4 8s
  - Shipping: £18 6s
  - Duty: £31 7s

If you need any further assistance, feel free to ask!
_____________________Parse Image into formattted_________________________
```json
{
  "invoice_number": "",
  "date": "May 1814",
  "total": "£327 15s 2d",
  "line items": "5",
  "items": [
    {"description": "Jaconet", "quantity": "67", "price": "£266"},
    {"description": "White Mull", "quantity": "90", "price": ""},
    {"description": "Fancy White Muslin", "quantity": "26", "price": ""},
    {"description": "Jonguil D", "quantity": "28", "price": ""},
    {"description": "Coquelicot D", "quantity": "55